# 메일 -> CSV

In [50]:
# 메일 분류
def classify_mail(subject, body):
    text = (subject + " " + body).lower()

    categories = {
        "업무협조": [
            "협조", "요청", "부탁", "전달", "회신", "확인", "처리", "검토", "승인", "진행", "지원", "일정 요청", "자료 요청", "공유 요청", "확인 부탁", "전달 부탁"
        ],
        "보고서": [
            "보고", "결과", "분석", "현황", "매출", "데이터", "통계", "성과", "개선", "지표", "리포트", "요약"
        ],
        "회의록": [
            "회의", "회의록", "안건", "참석자", "일정", "논의", "결정", "진행", "의결", "회의 결과"
        ],
        "공지": [
            "공지", "안내", "알림", "변경", "일정", "공지사항", "배포", "적용", "공지드립니다", "참고"
        ]
    }

    scores = {key: 0 for key in categories}

    for category, keywords in categories.items():
        for kw in keywords:
            if kw in text:
                scores[category] += 1

    # 점수 가장 높은 카테고리 선택
    best_category = max(scores, key=scores.get)

    # 전부 0이면 fallback
    if scores[best_category] == 0:
        return "공지"  # 기본값 (가장 범용적)

    return best_category

In [51]:
# 액션 분류
def detect_action(body):
    need_reply_keywords = [
        "회신", "답변", "의견 부탁", "피드백", "확인 부탁",
        "검토 후 회신", "회신 요청", "답장"
    ]

    # no_reply_keywords = [
    #     "참고", "공유드립니다", "공지", "안내", "전달드립니다"
    # ]

    for kw in need_reply_keywords:
        if kw in body:
            return "회신 필요"

    # for kw in no_reply_keywords:
    #     if kw in body:
    #         return "회신 불필요"

    return "회신 불필요"

In [52]:
# 기한 추출
def extract_deadline(body):
    patterns = [
        r"\d{4}[.-]\d{2}[.-]\d{2}",
        r"\d{2}[.-]\d{2}",
        r"\d{1,2}월\s?\d{1,2}일",
        r"\d{1,2}/\d{1,2}"
    ]

    for pattern in patterns:
        match = re.search(pattern, body)
        if match:
            return match.group()

    # "까지", "마감" 같은 문맥 기반 탐색
    deadline_keywords = ["까지", "마감", "기한"]
    for line in body.split("\n"):
        if any(k in line for k in deadline_keywords):
            return line.strip()

    return "-"

In [53]:
import imaplib
import email
from email.header import decode_header
import os
import pdfplumber
IMAP_SERVER = "imap.gmail.com"
EMAIL = 'mhnkms8041@gmail.com'
PASSWORD = 'qxvh ihup erwd ppgn'
SAVE_DIR = "./downloads"
os.makedirs(SAVE_DIR, exist_ok=True)
# 1. IMAP 접속
mail = imaplib.IMAP4_SSL(IMAP_SERVER)
mail.login(EMAIL, PASSWORD)
mail.select("inbox")

('OK', [b'19'])

In [54]:
# 2. 메일 검색 (제목 필터는 직접 처리)
result, data = mail.search(None, "ALL")
mail_ids = data[0].split()
mail_csv_list = []
target_mail = None
# 최근 메일부터 역순 탐색
for mail_id in reversed(mail_ids):
    result, data = mail.fetch(mail_id, "(RFC822)")
    raw_email = data[0][1]
    msg = email.message_from_bytes(raw_email)

    subject, encoding = decode_header(msg["Subject"])[0]
    if isinstance(subject, bytes):
        subject = subject.decode(encoding or "utf-8")

    # 필터 없이 전부 저장하거나 조건 유지 가능
    mail_csv_list.append(msg)
    # if "[업무협조]" in subject:
    #     print("대상 메일:", subject)
    #     mail_csv_list.append(msg)

In [55]:
import csv
import re
import shutil

csv_path = "email_list.csv"
file_exists = os.path.isfile(csv_path)

with open(csv_path, "a", newline="", encoding="utf-8-sig") as csvfile:
    writer = csv.writer(csvfile)

    if not file_exists:
        # writer.writerow(["제목", "발신자", "분류", "액션", "기한", "요약"])
        writer.writerow(["제목", "발신자", "분류", "액션", "기한"])

    for mail_item in mail_csv_list:

        subject, encoding = decode_header(mail_item["Subject"])[0]
        if isinstance(subject, bytes):
            subject = subject.decode(encoding or "utf-8")

        from_ = str(email.header.make_header(decode_header(mail_item.get("From"))))

        full_text = ""
        saved_files = []   # ← 파일 경로 저장용

        for part in mail_item.walk():
            content_disposition = part.get("Content-Disposition")

            if part.get_content_type() == "text/plain" and "attachment" not in str(content_disposition):
                body = part.get_payload(decode=True)
                if body:
                    full_text += body.decode("utf-8", errors="ignore") + "\n"

            if content_disposition and "attachment" in content_disposition:
                filename = part.get_filename()

                if filename:
                    filename, enc = decode_header(filename)[0]
                    if isinstance(filename, bytes):
                        filename = filename.decode(enc or "utf-8")

                    if filename.lower().endswith(".pdf"):

                        # temp 폴더에 먼저 저장
                        temp_dir = os.path.join(SAVE_DIR, "temp")
                        os.makedirs(temp_dir, exist_ok=True)

                        temp_path = os.path.join(temp_dir, filename)

                        with open(temp_path, "wb") as f:
                            f.write(part.get_payload(decode=True))

                        saved_files.append(temp_path)

                        # PDF 텍스트 추출
                        with pdfplumber.open(temp_path) as pdf:
                            for page in pdf.pages:
                                text = page.extract_text()
                                if text:
                                    full_text += text + "\n"

        # 최종 분류 (PDF 기반)
        category = classify_mail(subject, full_text)

        # 폴더 생성
        category_dir = os.path.join(SAVE_DIR, category)
        os.makedirs(category_dir, exist_ok=True)

        # 파일 이동
        for temp_path in saved_files:
            filename = os.path.basename(temp_path)
            final_path = os.path.join(category_dir, filename)

            shutil.move(temp_path, final_path)
            #print(f"이동 완료: {final_path}")

        if os.path.exists(temp_dir) and not os.listdir(temp_dir):
            os.rmdir(temp_dir)

        # 분석
        action = detect_action(full_text)
        deadline = extract_deadline(full_text)

        writer.writerow([
            subject,
            from_,
            category,
            action,
            deadline
        ])

        #print(f"처리 완료: {subject}")

print(f"\nCSV 저장 완료: {csv_path}")


CSV 저장 완료: email_list.csv
